# Download GSE139555 and build initial AnnData

Pulls the raw cellranger MTX matrices and the T cell metadata from GEO, merges them into a single `AnnData` keyed by cell barcode, and writes `data/raw_combined.h5ad` for use by notebook 02.

## 1. Colab setup

In [ ]:
# colab setup
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
PROJECT_DIR_DRIVE = '/content/drive/MyDrive/final_project'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_DIR = PROJECT_DIR_DRIVE
    subprocess.run(['pip', 'install', '-q', 'scanpy', 'umap-learn'], check=True)
else:
    # for running locally, assume notebook lives in notebooks/, project root is one up
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'models'), exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR    =', DATA_DIR)
print('RESULTS_DIR =', RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR = /content/drive/MyDrive/final_project
DATA_DIR    = /content/drive/MyDrive/final_project/data
RESULTS_DIR = /content/drive/MyDrive/final_project/results


In [ ]:
from pathlib import Path
from src import data as D

raw_tar = Path(DATA_DIR) / 'GSE139555_RAW.tar'
meta_gz = Path(DATA_DIR) / 'GSE139555_tcell_metadata.txt.gz'
raw_dir = Path(DATA_DIR) / 'raw'

D.download_file(D.RAW_TAR_URL, raw_tar)
D.download_file(D.TCELL_META_URL, meta_gz)
D.extract_tar(raw_tar, raw_dir)

[skip] /content/drive/MyDrive/final_project/data/GSE139555_RAW.tar already exists (638.4 MB)
[skip] /content/drive/MyDrive/final_project/data/GSE139555_tcell_metadata.txt.gz already exists (3.9 MB)
[extract] /content/drive/MyDrive/final_project/data/GSE139555_RAW.tar -> /content/drive/MyDrive/final_project/data/raw


PosixPath('/content/drive/MyDrive/final_project/data/raw')

### 2. Sanity-check the extracted files
Cellranger output should appear as `<GSM_id>_<sample>_{barcodes,features,matrix}` per sample.

In [ ]:
prefixes = D.list_sample_prefixes(raw_dir)
print(f'{len(prefixes)} samples found.')
for p in prefixes[:6]:
    print(' ', p)
if len(prefixes) > 6:
    print('  ...')

32 samples found.
  GSM4143655_SAM24345862-lt1
  GSM4143656_SAM24345863-ln1
  GSM4143657_SAM24348188-lt2
  GSM4143658_SAM24348189-ln2
  GSM4143659_SAM24349905-lt3
  GSM4143660_SAM24349906-ln3
  ...


### 3. Load T cell metadata
Visualize the columns to decide which to use for clonotype label

In [ ]:
meta = D.load_tcell_metadata(meta_gz)
print(meta.shape)
meta.head()

[meta] 141623 cells, columns: ['UMAP_1', 'UMAP_2', 'ident', 'patient', 'sample', 'source', 'clonotype']
(141623, 7)


,UMAP_1,UMAP_2,ident,patient,sample,source,clonotype
LT1_AAACCTGAGGATATAC-1,0.761298,1.301195,8.3a-Trm,Lung1,LT1,Tumor,lung1.tn.C1
LT1_AAACCTGAGTTACCCA-1,-6.475698,0.690571,4.3-TCF7,Lung1,LT1,Tumor,lung1.tn.C3
LT1_AAACCTGCAACACCCG-1,-1.326519,0.730108,4.4-FOS,Lung1,LT1,Tumor,lung1.tn.C5
LT1_AAACCTGCATCTCCCA-1,-3.510637,1.008227,4.4-FOS,Lung1,LT1,Tumor,lung1.tn.C8
LT1_AAACCTGGTTCGTCTC-1,-5.588617,1.632665,4.3-TCF7,Lung1,LT1,Tumor,lung1.tn.C12


In [ ]:
# show column dtypes and unique-value counts to pick clonotype/patient columns
for c in meta.columns:
    n = meta[c].nunique()
    print(f'{c:>25s}  nunique={n:<6d}  sample={list(meta[c].dropna().unique()[:3])}')

                   UMAP_1  nunique=141442  sample=[np.float64(0.761297941207886), np.float64(-6.47569751739502), np.float64(-1.32651913166046)]
                   UMAP_2  nunique=141455  sample=[np.float64(1.30119514465332), np.float64(0.690570771694183), np.float64(0.730108439922333)]
                    ident  nunique=16      sample=['8.3a-Trm', '4.3-TCF7', '4.4-FOS']
                  patient  nunique=14      sample=['Lung1', 'Lung2', 'Lung3']
                   sample  nunique=32      sample=['LT1', 'LN1', 'LT2']
                   source  nunique=3       sample=['Tumor', 'NAT', 'Blood']
                clonotype  nunique=55260   sample=['lung1.tn.C1', 'lung1.tn.C3', 'lung1.tn.C5']


### 4. Load all samples and concatenate

In [ ]:
import importlib
from src import data as D
importlib.reload(D)

adata = D.load_all_samples(raw_dir)
print(adata)

[load] 32 samples


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  GSM4143655_SAM24345862-lt1: 7462 cells x 30727 genes


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  GSM4143656_SAM24345863-ln1: 6169 cells x 30727 genes


...

  GSM4143684_SAM24363036-rt3: 6724 cells x 30727 genes


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  GSM4143685_SAM24363037-rn3: 5603 cells x 30727 genes


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  GSM4143686_SAM24363038-rb3: 3953 cells x 30727 genes
AnnData object with n_obs × n_vars = 200626 × 30727
    obs: 'sample'
    var: 'gene_id'


### 5. Attach metadata + save

In [ ]:
adata = D.attach_metadata(adata, meta)
print(adata)
print(adata.obs.head())

[merge] 141623 / 200626 cells matched metadata (metadata has 141623 total)
AnnData object with n_obs × n_vars = 141623 × 30727
    obs: 'sample', 'UMAP_1', 'UMAP_2', 'ident', 'patient', 'source', 'clonotype'
    var: 'gene_id'
                       sample    UMAP_1    UMAP_2     ident patient source  \
LT1_AAACCTGAGGATATAC-1    LT1  0.761298  1.301195  8.3a-Trm   Lung1  Tumor   
LT1_AAACCTGAGTTACCCA-1    LT1 -6.475698  0.690571  4.3-TCF7   Lung1  Tumor   
LT1_AAACCTGCAACACCCG-1    LT1 -1.326519  0.730108   4.4-FOS   Lung1  Tumor   
LT1_AAACCTGCATCTCCCA-1    LT1 -3.510637  1.008227   4.4-FOS   Lung1  Tumor   
LT1_AAACCTGGTTCGTCTC-1    LT1 -5.588617  1.632665  4.3-TCF7   Lung1  Tumor   

                           clonotype  
LT1_AAACCTGAGGATATAC-1   lung1.tn.C1  
LT1_AAACCTGAGTTACCCA-1   lung1.tn.C3  
LT1_AAACCTGCAACACCCG-1   lung1.tn.C5  
LT1_AAACCTGCATCTCCCA-1   lung1.tn.C8  
LT1_AAACCTGGTTCGTCTC-1  lung1.tn.C12  


In [ ]:
out = Path(DATA_DIR) / 'raw_combined.h5ad'
adata.write_h5ad(out, compression='gzip')
print('wrote', out, f'({out.stat().st_size/1e6:.1f} MB)')

wrote /content/drive/MyDrive/final_project/data/raw_combined.h5ad (304.7 MB)
